In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
from torch.optim import Adam
from torchvision.datasets import QMNIST

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

run_name = 'QMNIST_Orthogonal'
num_epochs = 50
batch_size = 64
learning_rate = 0.001

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = QMNIST(root='../../Final Benchmarks/QMNIST/data', train=True, download=True, transform=transform)
test_dataset = QMNIST(root='../../Final Benchmarks/QMNIST/data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
# No separate validation split in the source notebook: the test set is
# evaluated every epoch and doubles as "val" throughout training.
val_loader = test_loader

def custom_init(model):
    relu_gain = nn.init.calculate_gain('relu')
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.orthogonal_(m.weight, gain=relu_gain)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)


# Custom 9-Layer CNN Architecture (5 conv + 3 FC)
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()

        self.layer1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.layer2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.layer3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.layer4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.layer5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        # Pass dummy input to calculate output shape
        dummy_input = torch.zeros(1, 1, 32, 32)
        x = self.pool(torch.relu(self.layer2(torch.relu(self.layer1(dummy_input)))))
        x = self.pool(torch.relu(self.layer4(torch.relu(self.layer3(x)))))
        x = torch.relu(self.layer5(x))
        self.flattened_size = x.view(1, -1).shape[1]

        self.fc1 = nn.Linear(self.flattened_size, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 10)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.pool(torch.relu(self.layer2(x)))
        x = torch.relu(self.layer3(x))
        x = self.pool(torch.relu(self.layer4(x)))
        x = torch.relu(self.layer5(x))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x


# Initialize model, apply custom initialization
model = CustomCNN().to(device)
custom_init(model)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)  # L2 Regularization
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    total, correct, train_loss = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)
    train_loss_list.append(train_loss / len(train_loader))

    model.eval()
    correct, total, val_loss = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    val_acc_list.append(val_acc)
    val_loss_list.append(val_loss / len(val_loader))

    scheduler.step()

    print(f"Epoch [{epoch + 1}/{num_epochs}], "
          f"Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%, "
          f"Train Loss: {train_loss_list[-1]:.4f}, Val Loss: {val_loss_list[-1]:.4f}")

duration = time.time() - start_time
print(f"\nTraining completed in {duration:.2f} seconds.")

train_loss_list_final = train_loss_list
train_acc_list_final = train_acc_list
val_loss_list_final = val_loss_list
val_acc_list_final = val_acc_list
test_acc = val_acc_list[-1]
print(f'Test Accuracy: {test_acc:.2f}%')

# Save metrics to files labelled with dataset + init method
with open(f'{run_name}_metrics.csv', 'w') as f:
    f.write('epoch,train_loss,train_acc,val_loss,val_acc\n')
    for i in range(len(train_loss_list)):
        f.write(f'{i+1},{train_loss_list[i]:.6f},{train_acc_list[i]:.4f},{val_loss_list[i]:.6f},{val_acc_list[i]:.4f}\n')

with open(f'{run_name}_test_accuracy.txt', 'w') as f:
    f.write(f'{run_name} Test Accuracy: {test_acc:.2f}%\n')

# Visualization
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_loss_list, label='Train Loss')
plt.plot(val_loss_list, label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_acc_list, label='Train Acc')
plt.plot(val_acc_list, label='Val Acc')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.suptitle(run_name)
plt.tight_layout()
plt.savefig(f'{run_name}_curves.png', dpi=150)
plt.show()
